In [38]:
import os
import pandas as pd

In [39]:
print(os.getcwd())

c:\Users\selin\Desktop\job-market-analysis


Setup & Load

In [40]:
df = pd.read_csv("data/raw/kumarijob_info.csv")

Profile raw data

In [41]:
df.head()

,title,company,location,job_url,scraped_date,category
0,Sales Executive,Classic Tech Pvt. Ltd.,"Chabahil, Suryabinayak, Golfutar,Kalanki",https://www.kumarijob.com/classictech-pvt-ltd/...,3/10/2026,it
1,Fiber Bike Rider,Classic Tech Pvt. Ltd.,"kalanki, thankot, balaju,lalitpur,thimi",https://www.kumarijob.com/classictech-pvt-ltd/...,3/10/2026,it
2,Senior Data Engineer,CloudFactory,"Kathmandu, Nepal",https://www.kumarijob.com/more-jobs/67578-seni...,3/10/2026,it
3,Mid-Level SEO Analyst,Varosa Technology,"Kathmandu, Nepal",https://www.kumarijob.com/more-jobs/67577-mid-...,3/10/2026,it
4,Business and Data Analyst,Abacus Insights,"Kathmandu, Nepal",https://www.kumarijob.com/more-jobs/67576-busi...,3/10/2026,it


In [42]:
df.dtypes

title           str
company         str
location        str
job_url         str
scraped_date    str
category        str
dtype: object

In [43]:
df.isnull().sum()

title            0
company          1
location        10
job_url          1
scraped_date     0
category         0
dtype: int64

Remove Duplicates


In [44]:
len(df)



188

In [45]:
df=df.drop_duplicates(subset=["title","company"])

In [46]:
len(df)

188

Remove junk rows 

In [47]:
JUNK_TITLES=[
    "similar openings",
    "simialar jobs",
    "you may also like",
    "featured jobs"

]
before=len(df)
df=df[~df["title"].str.lower().str.strip().isin(JUNK_TITLES)]
after=len(df)

The ~ operator means "NOT" in pandas — it inverts the filter. df[condition] keeps matching rows, df[~condition] keeps everything else.

In [48]:
print(f"Removed{before-after} junk rows")
print(f"Remaining:{after}rows")

Removed1 junk rows
Remaining:187rows


Remocce Duplicate


In [49]:
before=len(df)

df=df.drop_duplicates(subset=["title","company"])

after=len(df)
print(f"Remaining:{after} rows")

Remaining:187 rows


In [50]:
import re 

In [51]:
def clean_title(title):
    if pd.isna(title):
        return "Unknown"
    
    title = str(title).strip()
    
    # Remove extra info after dash
    # "Data Center Technician - NP - Lalitpur - On-site" → "Data Center Technician"
    title = re.sub(r"\s*-\s*(NP|Nepal|Remote|On-site|Onsite|Hybrid).*", 
                   "", title, flags=re.IGNORECASE)
    
    # Remove content inside brackets
    # "Credit Analyst (CA)" → "Credit Analyst"
    title = re.sub(r"\s*\(.*?\)", "", title)
    
    # Normalize seniority prefixes
    title = re.sub(r"\bSr\.?\b", "Senior", title, flags=re.IGNORECASE)
    title = re.sub(r"\bJr\.?\b", "Junior", title, flags=re.IGNORECASE)
    
    # Title case
    title = title.strip().title()
    
    return title

df["title_clean"] = df["title"].apply(clean_title)

# Show what changed
changed = df[df["title"] != df["title_clean"]][["title", "title_clean"]]
print(f"Titles cleaned: {len(changed)}")
display(changed.head(15))

Titles cleaned: 65


,title,title_clean
3,Mid-Level SEO Analyst,Mid-Level Seo Analyst
4,Business and Data Analyst,Business And Data Analyst
7,Data Center Technician - NP - Lalitpur - On-site,Data Center Technician
8,Data Center Technician - NP - Thapathali - On-...,Data Center Technician
9,Data Center Technician - NP - Kathmandu - On-site,Data Center Technician
11,SEO Expert,Seo Expert
13,Senior SEO Specialist / Executive,Senior Seo Specialist / Executive
14,Software QA Engineer 2,Software Qa Engineer 2
16,Data and Platform Manager,Data And Platform Manager
17,Finance and Operation Officer,Finance And Operation Officer


Clean Company Names

In [52]:
def clean_company(company):
    if pd.isna(company) or company == 'N/A':
        return 'Unknown'
    
    company = str(company).strip()
    
    # Standardize Pvt. Ltd. variations
    company = re.sub(r'\bPvt\.?\s*Ltd\.?\b', 'Pvt. Ltd.', company, flags=re.IGNORECASE)
    company = re.sub(r'\bPrivate Limited\b', 'Pvt. Ltd.', company, flags=re.IGNORECASE)
    
    return company.strip()

df['company_clean'] = df['company'].apply(clean_company)

changed = df[df['company'] != df['company_clean']][['company', 'company_clean']]
print(f'Companies modified: {len(changed)}')
display(changed.drop_duplicates().head(15))

Companies modified: 44


,company,company_clean
0,Classic Tech Pvt. Ltd.,Classic Tech Pvt. Ltd..
13,Web Royale Pvt. Ltd.,Web Royale Pvt. Ltd..
17,Baniya Nirman Sewa Pvt. Ltd.,Baniya Nirman Sewa Pvt. Ltd..
18,E-Digital Nepal Pvt. Ltd.,E-Digital Nepal Pvt. Ltd..
36,ABC Healthcare Solutions Pvt. Ltd,ABC Healthcare Solutions Pvt. Ltd.
40,Nepal HR Solution PVt. Ltd.,Nepal HR Solution Pvt. Ltd..
41,GC International Group Pvt. Ltd.,GC International Group Pvt. Ltd..
43,Namaste A To Z Nepal Pvt. Ltd.,Namaste A To Z Nepal Pvt. Ltd..
44,Mandala Sutra and Consultants Pvt. Ltd.,Mandala Sutra and Consultants Pvt. Ltd..
48,Neasia Holidays Pvt. Ltd.,Neasia Holidays Pvt. Ltd..


In [53]:
LOCATION_MAP = {
    'ktm':                  'Kathmandu',
    'new baneswor':         'Kathmandu',
    'new baneshwor':        'Kathmandu',
    'baneswor':             'Kathmandu',
    'baneshwor':            'Kathmandu',
    'tinkune':              'Kathmandu',
    'naxal':                'Kathmandu',
    'lazimpat':             'Kathmandu',
    'chabahil':             'Kathmandu',
    'baluwatar':            'Kathmandu',
    'basundhara':           'Kathmandu',
    'gyaneshwor':           'Kathmandu',
    'sinamangal':           'Kathmandu',
    'tripureshwor':         'Kathmandu',
    'thamel':               'Kathmandu',
    'bagmati':              'Kathmandu',
    'bagmati province':     'Kathmandu',
    'kathmandu district':   'Kathmandu',
    'kathmandu, nepal':     'Kathmandu',
    'kupandole':            'Lalitpur',
    'patan':                'Lalitpur',
    'lalitpur, nepal':      'Lalitpur',
    'pokhara, nepal':       'Pokhara',
    'bharatpur':            'Chitwan',
    'nepal':                'Nepal (Remote)',
      'chamati':              'Kathmandu',
    'gyaneshwor tower':     'Kathmandu',
    'rb tower lazimpat':    'Kathmandu',
    'rb tower':             'Kathmandu',
    'maharajgunj':          'Kathmandu',
    'machhapokhari':        'Kathmandu',
    'shantinagar':          'Kathmandu',
    'anamnagar':            'Kathmandu',
    'thapagau baneswor':    'Kathmandu',
    'shantinagar gate':     'Kathmandu',
    'bluebird complex':     'Kathmandu',
    'new plaza':            'Kathmandu',
    'bishalnagar-4':        'Kathmandu',
    'suryabinayak':         'Bhaktapur',
    'madhyapur thimi':      'Bhaktapur',
    'koshi province':       'Biratnagar',
    'sudurpaschim':         'Dhangadhi',
    'lumbini':              'Butwal',
    'all over nepal':       'Nepal (Remote)',
}

KNOWN_CITIES = [
    'Kathmandu', 'Lalitpur', 'Bhaktapur', 'Pokhara',
    'Biratnagar', 'Butwal', 'Chitwan', 'Bharatpur',
    'Hetauda', 'Dharan', 'Birgunj', 'Nepalgunj',
    'Dhangadhi', 'Itahari', 'Parwanipur'
]

def clean_location(location):
    if pd.isna(location) or location == 'N/A':
        return 'Unknown'
    
    location = str(location).strip()
    
    # Take first city if multiple listed
    first = re.split(r'[,/]', location)[0].strip()
    
    # Check exact mapping
    mapped = LOCATION_MAP.get(first.lower())
    if mapped:
        return mapped
    
    # Check if a known city is mentioned anywhere in the string
    for city in KNOWN_CITIES:
        if city.lower() in location.lower():
            return city
    
    return first.title()

df['location_clean'] = df['location'].apply(clean_location)

print('=== LOCATION DISTRIBUTION AFTER CLEANING ===')
display(df['location_clean'].value_counts())

=== LOCATION DISTRIBUTION AFTER CLEANING ===


location_clean
Kathmandu                        122
Nepal (Remote)                    17
Lalitpur                          14
Unknown                            9
Butwal                             6
Biratnagar                         3
Bhaktapur                          3
Pokhara                            3
Chitwan                            1
Parwanipur                         1
Gyaneshwor & Kalanki               1
Dhangadhi                          1
Itahari                            1
Baglung                            1
Bharatpur                          1
Rb Tower Lazimpat (6Th Floor)      1
Birgunj                            1
Jhapa District                     1
Name: count, dtype: int64

Fix Category

In [54]:
def classify_category(title):
    title=str(title).lower()

    if any(k in title for k in ['data', 'sql', 'python', 'developer', 'software',
        'engineer', 'it ', 'tech', 'system', 'network',
        'cyber', 'cloud', 'devops', 'qa', 'java', 'php',
        'fullstack', 'full stack', 'backend', 'frontend',
        'linux', 'server', 'database', 'programmer',
        'web developer', 'mobile', 'android', 'ios']):
        return 'tech'
    
    elif any(k in title for k in [
        'account', 'finance', 'financial', 'audit',
        'tax', 'payroll', 'budget', 'credit', 'billing',
        'bookkeeper', 'treasurer', 'cost', 'revenue',
        'portfolio', 'pricing', 'mis officer'
    ]):
        return 'accounting'
    
    elif any(k in title for k in [
        'seo', 'sem', 'digital marketing', 'social media',
        'content creator', 'paid ads', 'email marketing',
        'google ads', 'ppc', 'media executive', 'paid media',
        'branding', 'graphic designer', 'video editor',
        'video content', 'content writer', 'copywriter',
        'editor', 'journalist', 'writer', 'blogger'
    ]):
        return 'digital_marketing'
    
    elif any(k in title for k in [
        'sales', 'marketing', 'business development',
        'brand', 'territory', 'field sales', 'trade',
        'dealer', 'distributor', 'retail', 'wholesale',
        'medical representative', 'sales manager',
        'sales officer', 'sales executive', 'sales rep'
    ]):
        return 'sales'
    
    elif any(k in title for k in [
        'civil', 'mechanical', 'electrical', 'architect',
        'structural', 'road', 'construction', 'drafter',
        'surveyor', 'site engineer', 'project engineer'
    ]):
        return 'engineering'

    elif any(k in title for k in [
        'hr', 'human resource', 'recruitment', 'talent',
        'people', 'training', 'learning', 'payroll'
    ]):
        return 'hr'

    elif any(k in title for k in [
        'customer service', 'customer support', 'customer relation',
        'call center', 'receptionist', 'front desk',
        'front office', 'help desk', 'support executive',
        'guest', 'client service'
    ]):
        return 'customer_service'

    elif any(k in title for k in [
        'logistics', 'supply chain', 'warehouse', 'driver',
        'rider', 'delivery', 'dispatch', 'fleet',
        'operations', 'procurement', 'purchasing'
    ]):
        return 'operations'

    else:
        return 'other'
    
df['category_clean']=df['title_clean'].apply(classify_category)
print('Cleaned categories')
comparision= pd.DataFrame({
    'Original': df['category'].value_counts(),
    'Cleaned' : df['category_clean'].value_counts()
    })
display(comparision)

print(comparision)

Cleaned categories


,Original,Cleaned
accounting,28.0,22.0
content_writing,8.0,NaN
customer_service,NaN,12.0
digital_marketing,65.0,42.0
engineering,7.0,2.0
fullstack,7.0,NaN
it,16.0,NaN
marketing,56.0,NaN
operations,NaN,2.0
other,NaN,12.0


                   Original  Cleaned
accounting             28.0     22.0
content_writing         8.0      NaN
customer_service        NaN     12.0
digital_marketing      65.0     42.0
engineering             7.0      2.0
fullstack               7.0      NaN
it                     16.0      NaN
marketing              56.0      NaN
operations              NaN      2.0
other                   NaN     12.0
sales                   NaN     68.0
tech                    NaN     27.0


In [55]:
df.head()

,title,company,location,job_url,scraped_date,category,title_clean,company_clean,location_clean,category_clean
0,Sales Executive,Classic Tech Pvt. Ltd.,"Chabahil, Suryabinayak, Golfutar,Kalanki",https://www.kumarijob.com/classictech-pvt-ltd/...,3/10/2026,it,Sales Executive,Classic Tech Pvt. Ltd..,Kathmandu,sales
1,Fiber Bike Rider,Classic Tech Pvt. Ltd.,"kalanki, thankot, balaju,lalitpur,thimi",https://www.kumarijob.com/classictech-pvt-ltd/...,3/10/2026,it,Fiber Bike Rider,Classic Tech Pvt. Ltd..,Lalitpur,operations
2,Senior Data Engineer,CloudFactory,"Kathmandu, Nepal",https://www.kumarijob.com/more-jobs/67578-seni...,3/10/2026,it,Senior Data Engineer,CloudFactory,Kathmandu,tech
3,Mid-Level SEO Analyst,Varosa Technology,"Kathmandu, Nepal",https://www.kumarijob.com/more-jobs/67577-mid-...,3/10/2026,it,Mid-Level Seo Analyst,Varosa Technology,Kathmandu,digital_marketing
4,Business and Data Analyst,Abacus Insights,"Kathmandu, Nepal",https://www.kumarijob.com/more-jobs/67576-busi...,3/10/2026,it,Business And Data Analyst,Abacus Insights,Kathmandu,tech


Add Derived Columns

In [56]:
def get_title(title):
    title=str(title).lower()
    if any(k in title for k in ['senior', 'sr', 'lead', 'head',
                                  'manager', 'director', 'principal']):
        return 'senior'
    elif any(k in title for k in ['junior', 'jr', 'trainee',
                                   'intern', 'associate', 'assistant']):
        return 'Junior'
    else:
        return 'Mid-Level'
    
def get_job_type(title):
    title = str(title).lower()
    if 'remote' in title:
        return 'Remote'
    elif 'hybrid' in title:
        return 'Hybrid'
    else:
        return 'On-site'
    
df['seniority']=df['title_clean'].apply(get_title)
df['job_type']  = df['title'].apply(get_job_type)

print('=== SENIORITY DISTRIBUTION ===')
display(df['seniority'].value_counts())
print('\n=== JOB TYPE DISTRIBUTION ===')
display(df['job_type'].value_counts())
    

=== SENIORITY DISTRIBUTION ===


seniority
Mid-Level    139
senior        34
Junior        14
Name: count, dtype: int64


=== JOB TYPE DISTRIBUTION ===


job_type
On-site    182
Remote       5
Name: count, dtype: int64

Build Final Clean DataFrame

In [57]:
df_clean = df[[
    'title_clean',
    'company_clean',
    'location_clean',
    'category_clean',
    'seniority',
    'job_type',
    'job_url',
    'scraped_date'
]].rename(columns={
    'title_clean':    'title',
    'company_clean':  'company',
    'location_clean': 'location',
    'category_clean': 'category'
})

# Replace N/A strings with proper NaN
df_clean = df_clean.replace('N/A', pd.NA)

print(f'Final shape: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns')
display(df_clean.head(15))

Final shape: 187 rows x 8 columns


,title,company,location,category,seniority,job_type,job_url,scraped_date
0,Sales Executive,Classic Tech Pvt. Ltd..,Kathmandu,sales,Mid-Level,On-site,https://www.kumarijob.com/classictech-pvt-ltd/...,3/10/2026
1,Fiber Bike Rider,Classic Tech Pvt. Ltd..,Lalitpur,operations,Mid-Level,On-site,https://www.kumarijob.com/classictech-pvt-ltd/...,3/10/2026
2,Senior Data Engineer,CloudFactory,Kathmandu,tech,senior,On-site,https://www.kumarijob.com/more-jobs/67578-seni...,3/10/2026
3,Mid-Level Seo Analyst,Varosa Technology,Kathmandu,digital_marketing,Mid-Level,On-site,https://www.kumarijob.com/more-jobs/67577-mid-...,3/10/2026
4,Business And Data Analyst,Abacus Insights,Kathmandu,tech,Mid-Level,On-site,https://www.kumarijob.com/more-jobs/67576-busi...,3/10/2026
5,Linux System Admin,PST.AG,Nepal (Remote),tech,Mid-Level,On-site,https://www.kumarijob.com/more-jobs/67575-linu...,3/10/2026
6,Service Operations Specialist,CloudFactory,Kathmandu,operations,Mid-Level,On-site,https://www.kumarijob.com/more-jobs/67574-serv...,3/10/2026
7,Data Center Technician,Reboot Monkey,Lalitpur,tech,Mid-Level,On-site,https://www.kumarijob.com/more-jobs/67573-data...,3/10/2026
8,Data Center Technician,Reboot Monkey,Nepal (Remote),tech,Mid-Level,On-site,https://www.kumarijob.com/more-jobs/67572-data...,3/10/2026
9,Data Center Technician,Reboot Monkey,Kathmandu,tech,Mid-Level,On-site,https://www.kumarijob.com/more-jobs/67571-data...,3/10/2026


Validate Clean Data
> Validation catches silent failures — if cleaning breaks something, you know immediately instead of loading bad data into your database.

In [58]:
print('=== VALIDATION CHECKS ===')

checks = {
    'No empty titles':      df_clean['title'].isna().sum() == 0,
    'No empty companies':   df_clean['company'].isna().sum() == 0,
    'No duplicate rows':    df_clean.duplicated(subset=['title', 'company']).sum() == 0,
    'All categories valid': df_clean['category'].isin([
                                'tech', 'accounting', 'sales_marketing',
                                'digital_marketing', 'engineering',
                                'content_writing', 'other'
                            ]).all(),
}

for check, result in checks.items():
    status = 'PASS' if result else 'FAIL'
    print(f'  {status}  {check}')

print('\n=== FINAL MISSING VALUES ===')
print(df_clean.isnull().sum())

=== VALIDATION CHECKS ===
  PASS  No empty titles
  PASS  No empty companies
  FAIL  No duplicate rows
  FAIL  All categories valid

=== FINAL MISSING VALUES ===
title           0
company         0
location        0
category        0
seniority       0
job_type        0
job_url         0
scraped_date    0
dtype: int64


In [59]:
df_clean.head()

,title,company,location,category,seniority,job_type,job_url,scraped_date
0,Sales Executive,Classic Tech Pvt. Ltd..,Kathmandu,sales,Mid-Level,On-site,https://www.kumarijob.com/classictech-pvt-ltd/...,3/10/2026
1,Fiber Bike Rider,Classic Tech Pvt. Ltd..,Lalitpur,operations,Mid-Level,On-site,https://www.kumarijob.com/classictech-pvt-ltd/...,3/10/2026
2,Senior Data Engineer,CloudFactory,Kathmandu,tech,senior,On-site,https://www.kumarijob.com/more-jobs/67578-seni...,3/10/2026
3,Mid-Level Seo Analyst,Varosa Technology,Kathmandu,digital_marketing,Mid-Level,On-site,https://www.kumarijob.com/more-jobs/67577-mid-...,3/10/2026
4,Business And Data Analyst,Abacus Insights,Kathmandu,tech,Mid-Level,On-site,https://www.kumarijob.com/more-jobs/67576-busi...,3/10/2026


Save Clean Data

In [60]:

# Get project root (one level up from etl/)
PROJECT_ROOT = os.path.dirname(os.path.abspath(os.getcwd()))


# If already at root, use current directory
if os.path.basename(os.getcwd()) == 'etl':
    os.chdir('..')

PROJECT_ROOT = os.getcwd()
print('Project root:', PROJECT_ROOT)

Project root: c:\Users\selin\Desktop\job-market-analysis


In [62]:
os.makedirs('data/processed',exist_ok=True)

output_path='data/processed/kumarijob_clean.csv'
df_clean.to_csv(output_path,index=False)
print(f'saved:{output_path}')

saved:data/processed/kumarijob_clean.csv
